# Lightweight Fine-Tuning Project

## NLP Tweets - GPT2 Probe
High Score on Kaggle Disaster Tweets x 25 epochs (V12) = 0.81091 
- https://www.kaggle.com/code/jamesmcguigan/nlp-tweets-gpt2-probe?scriptVersionId=235316172

TODO: In this cell, describe your choices for each of the following

* PEFT technique: GPT2ForSequenceClassification + get_peft_model(LoraConfig)
* Model: GPT2
* Evaluation approach: trainer.evaluate() + Kaggle Leaderboard
* Fine-tuning dataset: Kaggle Disaster Tweets

In [1]:
!pip show transformers

Name: transformers
Version: 4.36.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/conda/lib/python3.10/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: auto-gptq, optimum, peft


## Loading and Evaluating a Foundation Model

> TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

Foundation Model chosen is "distilbert-base-uncased" instead of "gpt2"
Foundation Model has the wrong output size for classification task
DistilBertForSequenceClassification wraps a Probing Classifier Head to Foundation Model

Training is to freeze `base_model` weights to preserve large Foundation Model Weights
Training AutoModelForSequenceClassification as Probe Head on IMDB dataset

```
GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)           # token embeddings
    (wpe): Embedding(1024, 768)            # positional embeddings
    (drop): Dropout(p=0.1)                 # input dropout
    (h): ModuleList([                      # 12× GPT2Block
      GPT2Block(
        (ln_1): LayerNorm(768)             # pre‐attention layer norm
        (attn): GPT2Attention(             # multi‑head self‑attention
          (c_attn): Conv1D(2304, 768)      # QKV projection
          (c_proj): Conv1D(768, 768)       # output projection
          (attn_dropout): Dropout(0.1)
          (resid_dropout): Dropout(0.1)
        )
        (ln_2): LayerNorm(768)             # post‑attention layer norm
        (mlp): GPT2MLP(                    # feed‑forward MLP
          (c_fc): Conv1D(3072, 768)        # first FC
          (c_proj): Conv1D(768, 3072)      # second FC
          (act): NewGELUActivation()       # GELU
          (dropout): Dropout(0.1)
        )
      )
      …  # repeated 12 times
    ])
    (ln_f): LayerNorm(768)                 # final layer norm
  )
  (score): Linear(768 → num_labels)        # your classification head
)
```

In [2]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import transformers 
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from datasets import load_dataset, Dataset, DatasetDict
# from sklearn.model_selection import train_test_split
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
id2label={0: "NEGATIVE",    1: "POSITIVE"}     
label2id={   "NEGATIVE": 0,    "POSITIVE": 1}

model_name = "gpt2"
model      = GPT2ForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    id2label=id2label, 
    label2id=label2id,
)
model.config.pad_token_id = model.config.eos_token_id
model.to(device)
base_model = model

tokenizer  = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})


# Parameter Efficient Fine-Tuning - only train probe head, not base model
for param in model.parameters():            param.requires_grad = True
for param in model.base_model.parameters(): param.requires_grad = False
model

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=2, bias=False)
)

# Dataset and Evaluation

In [4]:
dataset = imdb = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
# Tokenize in batch mode (recommended)
def tokenize_function(batch):
    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True)
    if 'target' in batch:
        tokenized['labels'] = batch['target']
        # tokenized["labels"] = [float(x) for x in batch["target"]] # Convert ints to floats for MSELoss
    return tokenized

dataset = imdb = load_dataset("imdb")
tokenized_dataset = DatasetDict({
    "train": imdb['train'       ].shuffle(seed=42).select(range(500)).map(tokenize_function, batched=True),
    "test":  imdb['test'        ].shuffle(seed=42).select(range(500)).map(tokenize_function, batched=True),
    "eval":  imdb['unsupervised'].shuffle(seed=42).select(range(500)).map(tokenize_function, batched=True),
})

# Evaluate Model before training Probe Head

Untrained Probe Head unable to distinguish between Awesome and Bogus!

In [6]:
%%time
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model=model, 
    tokenizer=tokenizer, 
    padding="max_length", 
    truncation=True, 
    device=device
)
print(sentiment_pipeline("Everything is awesome!"), '->', "Everything is Awesome!")
print(sentiment_pipeline("Everything is bogus!"),   '->', "Everything is Bogus!")

[{'label': 'POSITIVE', 'score': 0.9846450686454773}] -> Everything is Awesome!
[{'label': 'POSITIVE', 'score': 0.9830958247184753}] -> Everything is Bogus!
CPU times: user 615 ms, sys: 195 ms, total: 810 ms
Wall time: 809 ms


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.
- https://learn.udacity.com/nanodegrees/nd608/parts/cd13303/lessons/786df5de-95ad-4e0d-be51-cc8a1c1e40fe/concepts/ed4cd691-b999-454e-b715-a603fb2aeeb5?lesson_tab=lesson
- https://huggingface.co/docs/peft/main/en/conceptual_guides/lora

In [7]:
from peft import get_peft_model, LoraConfig

# 2) Configure LoRA—only inject trainable rank‑r matrices into attention & MLP
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn","c_proj"],  # you can also target "mlp.c_fc", etc.
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)

# 3) Wrap model
peft_model = get_peft_model(base_model, lora_cfg)
peft_model

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): GPT2ForSequenceClassification(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): Linear(
                in_features=768, out_features=2304, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding

In [8]:
peft_model.print_trainable_parameters()

trainable params: 814,080 || all params: 125,253,888 || trainable%: 0.6499438963523432


In [9]:
%%time
# LINT-FIX: You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True

import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": (predictions == labels).mean()
    }


# The HuggingFace Trainer class handles the training and eval loop for PyTorch for us.
# Read more about it here https://huggingface.co/docs/transformers/main_classes/trainer
trainer = Trainer(
    model = peft_model,
    args  = TrainingArguments(
        output_dir = "./data/sentiment_analysis",
        # learning_rate = 2e-3,              # Udacity
        learning_rate = 5e-4,               # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        lr_scheduler_type = "linear",       # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        warmup_steps = 100,                 # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        
        per_device_train_batch_size = 4,  # Reduce the batch size if you don't have enough memory
        per_device_eval_batch_size  = 4,
        num_train_epochs = 5, 
        weight_decay = 0.01,
        # evaluation_strategy="epoch",
        # save_strategy="epoch",
        # load_best_model_at_end=True,
        report_to="none"
    ),
    train_dataset   = tokenized_dataset["train"],
    eval_dataset    = tokenized_dataset["eval"],
    tokenizer       = tokenizer,
    data_collator   = DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics = compute_metrics,
)

CPU times: user 172 ms, sys: 1.88 ms, total: 173 ms
Wall time: 172 ms


In [10]:
%%time
trainer.train()

Step,Training Loss


CPU times: user 1min 53s, sys: 176 ms, total: 1min 53s
Wall time: 1min 53s


TrainOutput(global_step=125, training_loss=1.0220242309570313, metrics={'train_runtime': 113.5731, 'train_samples_per_second': 4.402, 'train_steps_per_second': 1.101, 'total_flos': 263792885760000.0, 'train_loss': 1.0220242309570313, 'epoch': 1.0})

## Save and Reload
###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [11]:
# Saving the model (unchanged)
peft_model.save_pretrained("/tmp/distilbert-peft")  # Save the model

# Reloading the model
from peft import AutoPeftModelForSequenceClassification
reloaded_model = AutoPeftModelForSequenceClassification.from_pretrained("/tmp/distilbert-peft")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
!ls -la /tmp/distilbert-peft/

total 3216
drwxr-xr-x 2 student student    4096 Jul 18 19:48 .
drwxrwxrwt 1 root    root       4096 Jul 18 20:15 ..
-rw-r--r-- 1 student student     153 Jul 18 20:17 README.md
-rw-r--r-- 1 student student     423 Jul 18 20:17 adapter_config.json
-rw-r--r-- 1 student student 3275724 Jul 18 20:17 adapter_model.bin


## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [13]:
trainer.evaluate()

/opt/conda/conda-bld/pytorch_1682343967769/work/aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343967769/work/aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343967769/work/aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343967769/work/aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
